# Week 13 Puzzles — Interactive Dashboards & Data Storytelling

> **NS5116 電腦硬體與程式語言在行為科學實驗與大數據分析之應用 — Spring 2026**

這份 notebook 對應 Week 13 講義。所有練習都使用本週的三個情境：Stroop / behavioral RT、PsyArXiv preprint metadata、教育部高教統計。

- **Part 1 — Guided Practice (Puzzles 1–10)：** 每題都有可執行的參考解答。
- **Part 2 — Independent Practice (Puzzles 11–20)：** 空白 code cell 留給課堂練習。

**注意**：每個 code cell 都獨立 import 所需 package，方便你單獨複製執行。


## Part 1 — Guided Practice (with solutions)


### Puzzle 1 — Bar Chart: Stroop 平均 RT

**任務**：模擬 congruent / incongruent 兩個 condition 的 RT，計算平均值，並用 Plotly Express 畫 horizontal bar chart。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

np.random.seed(42)
df = pd.DataFrame({
    "condition": ["congruent"] * 60 + ["incongruent"] * 60,
    "rt_ms": np.concatenate([
        np.random.normal(455, 55, 60),
        np.random.normal(535, 75, 60),
    ]),
})

summary = (df.groupby("condition", as_index=False)["rt_ms"]
             .mean()
             .sort_values("rt_ms"))

fig = px.bar(
    summary,
    x="rt_ms",
    y="condition",
    orientation="h",
    color="rt_ms",
    color_continuous_scale="Blues",
    title="Mean RT by Stroop condition",
    labels={"rt_ms": "Mean RT (ms)", "condition": "Condition"},
)
fig.update_layout(coloraxis_showscale=False)
fig.show()


### Puzzle 2 — Line Chart: Trial-by-trial RT + Reference Line

**任務**：畫出 80 個 trial 的 RT，並加入 `median RT` 的 horizontal reference line。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

np.random.seed(42)
df = pd.DataFrame({
    "trial": np.arange(1, 81),
    "rt_ms": np.random.normal(510, 85, 80).clip(250, 900),
})
median_rt = df["rt_ms"].median()

fig = px.line(
    df,
    x="trial",
    y="rt_ms",
    markers=True,
    title="Trial-by-trial RT with median reference",
    labels={"trial": "Trial", "rt_ms": "RT (ms)"},
)
fig.add_hline(
    y=median_rt,
    line_dash="dot",
    line_color="orange",
    annotation_text=f"Median RT = {median_rt:.0f} ms",
    annotation_position="top right",
)
fig.update_layout(hovermode="x unified")
fig.show()


### Puzzle 3 — Scatter Plot: Conflict Score vs. RT

**任務**：模擬 trial-level conflict score，畫出 conflict score 與 RT 的 scatter plot，並用 `hover_data` 顯示 condition 與 accuracy。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

np.random.seed(42)
n = 120
condition = np.random.choice(["congruent", "incongruent"], size=n, p=[0.5, 0.5])
conflict = np.where(condition == "incongruent", np.random.normal(0.75, 0.15, n), np.random.normal(0.25, 0.12, n))
rt_ms = 430 + conflict * 180 + np.random.normal(0, 45, n)
accuracy = np.random.binomial(1, np.where(condition == "incongruent", 0.82, 0.93))

df = pd.DataFrame({"condition": condition, "conflict": conflict, "rt_ms": rt_ms, "accuracy": accuracy})

fig = px.scatter(
    df,
    x="conflict",
    y="rt_ms",
    color="condition",
    hover_data={"accuracy": True, "conflict": ":.2f", "rt_ms": ":.1f"},
    title="Conflict score vs. RT",
    labels={"conflict": "Conflict score", "rt_ms": "RT (ms)"},
    opacity=0.75,
)
fig.show()


### Puzzle 4 — Annotation: 標出最慢 trial

**任務**：在 trial-by-trial RT 圖上找出最慢 trial，用 annotation 標出來。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

np.random.seed(7)
df = pd.DataFrame({
    "trial": np.arange(1, 61),
    "rt_ms": np.random.normal(500, 70, 60).clip(250, 850),
})
df.loc[44, "rt_ms"] = 920  # intentional slow trial

peak_idx = df["rt_ms"].idxmax()
peak_trial = df.loc[peak_idx, "trial"]
peak_rt = df.loc[peak_idx, "rt_ms"]

fig = px.line(df, x="trial", y="rt_ms", markers=True, title="RT outlier check")
fig.add_annotation(
    x=peak_trial,
    y=peak_rt,
    text=f"Slowest trial: {peak_rt:.0f} ms",
    showarrow=True,
    arrowhead=2,
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="red",
)
fig.show()


### Puzzle 5 — Grouped Bar: Age Group × Condition

**任務**：比較 young / older adults 在 congruent / incongruent 兩種 condition 下的平均 RT，使用 grouped bar chart。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

np.random.seed(42)
rows = []
for age_group, base in [("young", 430), ("older", 520)]:
    for condition, cost in [("congruent", 0), ("incongruent", 85)]:
        rts = np.random.normal(base + cost, 55, 50)
        for rt in rts:
            rows.append({"age_group": age_group, "condition": condition, "rt_ms": rt})

df = pd.DataFrame(rows)
summary = df.groupby(["age_group", "condition"], as_index=False)["rt_ms"].mean()

fig = px.bar(
    summary,
    x="age_group",
    y="rt_ms",
    color="condition",
    barmode="group",
    title="Mean RT by age group and Stroop condition",
    labels={"age_group": "Age group", "rt_ms": "Mean RT (ms)", "condition": "Condition"},
)
fig.show()


### Puzzle 6 — Multi-Line Chart: MOE Trend by Sector

**任務**：用合成的高教統計資料畫出 public / private sector 的逐年學生數變化。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

np.random.seed(42)
years = np.arange(105, 114)
df = pd.DataFrame({
    "學年度": list(years) * 2,
    "sector": ["公立"] * len(years) + ["私立"] * len(years),
    "students": np.concatenate([
        np.linspace(420000, 395000, len(years)) + np.random.normal(0, 2500, len(years)),
        np.linspace(760000, 590000, len(years)) + np.random.normal(0, 4500, len(years)),
    ]).round().astype(int),
})

fig = px.line(
    df,
    x="學年度",
    y="students",
    color="sector",
    markers=True,
    title="Synthetic MOE trend — public vs. private",
    labels={"students": "Students", "sector": "Sector"},
)
fig.update_layout(hovermode="x unified")
fig.show()


### Puzzle 7 — GroupBy Preparation: PsyArXiv Top Subjects

**任務**：從 preprint metadata 中計算 top subjects，並畫 horizontal bar chart。


In [ ]:
import pandas as pd
import plotly.express as px

df = pd.DataFrame({
    "primary_subject": [
        "Cognitive Neuroscience", "Clinical Psychology", "Cognitive Neuroscience",
        "Developmental Psychology", "Social Psychology", "Cognitive Neuroscience",
        "Clinical Psychology", "Meta-science", "Social Psychology", "Clinical Psychology",
    ],
    "n_tags": [4, 2, 5, 3, 6, 1, 0, 7, 4, 3],
})

counts = (df["primary_subject"].value_counts()
            .rename_axis("subject")
            .reset_index(name="n"))

fig = px.bar(
    counts,
    x="n",
    y="subject",
    orientation="h",
    title="Top PsyArXiv subjects in sample metadata",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()


### Puzzle 8 — Histogram: RT Distribution

**任務**：畫出 RT distribution，並用 color 比較 congruent / incongruent。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

np.random.seed(42)
df = pd.DataFrame({
    "condition": ["congruent"] * 100 + ["incongruent"] * 100,
    "rt_ms": np.concatenate([
        np.random.normal(455, 55, 100),
        np.random.normal(535, 75, 100),
    ]),
})

fig = px.histogram(
    df,
    x="rt_ms",
    color="condition",
    nbins=30,
    barmode="overlay",
    opacity=0.65,
    title="RT distribution by Stroop condition",
    labels={"rt_ms": "RT (ms)"},
)
fig.show()


### Puzzle 9 — Altair Line Chart

**任務**：用 Altair 重畫 Puzzle 6 的 sector trend。


In [ ]:
import altair as alt
import numpy as np
import pandas as pd

np.random.seed(42)
years = np.arange(105, 114)
df = pd.DataFrame({
    "學年度": list(years) * 2,
    "sector": ["公立"] * len(years) + ["私立"] * len(years),
    "students": np.concatenate([
        np.linspace(420000, 395000, len(years)),
        np.linspace(760000, 590000, len(years)),
    ]).round().astype(int),
})

chart = (
    alt.Chart(df)
       .mark_line(point=True)
       .encode(
           x=alt.X("學年度:O", title="Academic year"),
           y=alt.Y("students:Q", title="Students"),
           color=alt.Color("sector:N", title="Sector"),
           tooltip=["學年度:O", "sector:N", "students:Q"],
       )
       .properties(title="MOE sector trend — Altair version", width=650, height=320)
       .interactive()
)
chart


### Puzzle 10 — Storytelling Caption

**任務**：把弱 caption 改成有解讀的 takeaway sentence。

弱 caption：`This is a line chart of students by year.`


In [ ]:
weak_caption = "This is a line chart of students by year."
strong_caption = (
    "私立大專學生數在 105–113 學年度間呈現更陡的下降，"
    "這支持少子化衝擊首先集中在私校的政策討論。"
)

print("Weak:", weak_caption)
print("Strong:", strong_caption)


---

## Part 2 — Independent Practice

下面 10 題請自己完成。每題都對應 Week 13 講義中的一個圖表或 storytelling 技巧。


### Puzzle 11 — Horizontal Bar Chart

用下列 subject counts 畫 horizontal bar chart，並讓最多的 subject 排在最上方。


In [ ]:
import pandas as pd
import plotly.express as px

counts = pd.DataFrame({
    "subject": ["Cognitive Neuroscience", "Clinical Psychology", "Developmental Psychology", "Social Psychology", "Meta-science"],
    "n": [48, 36, 31, 28, 22],
})

# 你的程式碼從這裡開始


### Puzzle 12 — Monthly Preprint Line Chart

用 `month` 與 `n_preprints` 畫 line chart，並設定 `hovermode="x unified"`。


In [ ]:
import pandas as pd
import plotly.express as px

monthly = pd.DataFrame({
    "month": pd.date_range("2026-01-01", periods=6, freq="MS"),
    "n_preprints": [92, 88, 103, 117, 96, 110],
})

# 你的程式碼從這裡開始


### Puzzle 13 — RT Histogram

模擬兩個 condition 的 RT，畫 overlay histogram。記得設定 `opacity`。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

np.random.seed(0)
df = pd.DataFrame({
    "condition": ["congruent"] * 80 + ["incongruent"] * 80,
    "rt_ms": np.concatenate([
        np.random.normal(460, 60, 80),
        np.random.normal(540, 80, 80),
    ]),
})

# 你的程式碼從這裡開始


### Puzzle 14 — Box Plot by Condition

用 Puzzle 13 的資料畫 box plot，比較兩個 condition 的 RT 分布。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

np.random.seed(1)
df = pd.DataFrame({
    "condition": ["congruent"] * 80 + ["incongruent"] * 80,
    "rt_ms": np.concatenate([
        np.random.normal(460, 60, 80),
        np.random.normal(540, 80, 80),
    ]),
})

# 你的程式碼從這裡開始


### Puzzle 15 — EEG Power Heatmap

用 `px.imshow()` 畫一張 channel × frequency band 的 power heatmap。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

np.random.seed(42)
channels = ["Fz", "Cz", "Pz", "Oz"]
bands = ["theta", "alpha", "beta", "gamma"]
power = np.random.normal(10, 2, (len(channels), len(bands)))

# 你的程式碼從這裡開始


### Puzzle 16 — Add Multiple Annotations

用 MOE sector trend 資料，標出 private sector 的起點與終點，讓讀者快速看到下降幅度。


In [ ]:
import pandas as pd
import plotly.express as px

df = pd.DataFrame({
    "學年度": list(range(105, 114)),
    "private_students": [760000, 742000, 720000, 695000, 670000, 645000, 625000, 606000, 590000],
})

# 你的程式碼從這裡開始


### Puzzle 17 — Faceted Charts

模擬 N-back task 中 load 1 / 2 / 3 的 RT，使用 `facet_col="load"` 畫 small multiples。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

np.random.seed(42)
rows = []
for load, mean_rt in [(1, 480), (2, 560), (3, 650)]:
    for rt in np.random.normal(mean_rt, 70, 70):
        rows.append({"load": str(load), "rt_ms": rt})
df = pd.DataFrame(rows)

# 你的程式碼從這裡開始


### Puzzle 18 — Customize Hover Data

建立 scatter plot，hover 時只顯示 title、date、subject，不顯示 raw index。


In [ ]:
import pandas as pd
import plotly.express as px

df = pd.DataFrame({
    "title": ["Working memory and aging", "Attention in visual search", "Prediction error in learning"],
    "date": pd.to_datetime(["2026-04-01", "2026-04-03", "2026-04-08"]),
    "subject": ["Cognitive Aging", "Attention", "Learning"],
    "n_tags": [4, 6, 3],
    "title_len": [24, 26, 28],
})

# 你的程式碼從這裡開始


### Puzzle 19 — Altair Conditional Color

用 Altair 畫 bar chart。若 `pct_change < -15`，bar 顯示橘色；否則顯示藍色。


In [ ]:
import altair as alt
import pandas as pd

df = pd.DataFrame({
    "city": ["臺北市", "新北市", "臺中市", "高雄市", "臺南市"],
    "pct_change": [-18.2, -9.5, -12.1, -20.4, -7.3],
})

# 你的程式碼從這裡開始


### Puzzle 20 — Complete Dashboard Figure Set (Bonus)

為一個 Streamlit dashboard 準備 3 張 Plotly figures：

1. Stroop mean RT bar chart
2. PsyArXiv top subject bar chart
3. MOE sector trend line chart

**加分**：每張圖下方寫一句 `caption`，句子必須是 takeaway，不是圖表描述。


In [ ]:
# 你的程式碼從這裡開始
